# CTGEN: Carbon Nanotube Generation

Generate carbon-nanotube candidates with the NTGEN diffusion model (SCIGEN-style
mask-constrained denoising, "Pathway 3" — no retraining required).

**Pipeline**: build a `SampleDataset` batch whose `SC_CarbonTube` constraint pins a
rolled-graphene CNT wall (chiral indices `(n, m)`) via `mask_x`/`mask_l`/`mask_t` →
run `sample_scigen` reverse diffusion (known wall atoms are re-imposed at every
step, the model denoises the rest) → convert to `pymatgen` structures → export CIF.

**Constraint keys** (`sc_dict` in `script/sc_utils.py`): `'cnt'` carbon nanotube
(full rolled-graphene wall), `'ntb'` generalized skeleton nanotube (any element),
`'van'` unconstrained vanilla.

**Checkpoint**: defaults to the public SCIGEN `mp_20` diffusion checkpoint
(auto-downloaded from Figshare). Once you train a carbon checkpoint (see the
retraining recipe at the bottom), point `MODEL_PATH` at it instead.

In [ ]:
# --- Environment & paths (local run, not Colab) -------------------------------
import os, sys
from pathlib import Path

# NTGEN-edit code tree: sibling of this notebook's folder (override via env).
PROJECT_DIR = Path(os.environ.get(
    'CTGEN_PROJECT_DIR',
    Path.cwd().resolve().parent / 'NTGEN-edit'
)).resolve()
assert PROJECT_DIR.exists(), f'NTGEN-edit not found at {PROJECT_DIR}'

# The package dir is named ntgent/ but all imports & hydra targets say scigen.*
# -> self-heal with a symlink if needed.
if not (PROJECT_DIR / 'scigen').exists() and (PROJECT_DIR / 'ntgent').exists():
    os.symlink('ntgent', PROJECT_DIR / 'scigen')
    print('created symlink scigen -> ntgent')

os.environ.setdefault('PROJECT_ROOT', str(PROJECT_DIR))
os.environ.setdefault('HYDRA_JOBS', str(PROJECT_DIR))
os.environ.setdefault('WANDB_DIR', str(PROJECT_DIR / 'wandb'))
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

for p in (str(PROJECT_DIR), str(PROJECT_DIR / 'script')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_DIR)   # gen_utils opens ./data/kde_bond.pkl relative to cwd
print('PROJECT_DIR:', PROJECT_DIR)

In [ ]:
# --- Diffusion checkpoint (public SCIGEN mp_20 weights from Figshare) ----------
import json, zipfile
from urllib.request import urlopen, urlretrieve

MODEL_PATH = PROJECT_DIR / 'models' / 'mp_20'
# >>> After training a carbon checkpoint (see recipe at the bottom), use e.g.:
# MODEL_PATH = PROJECT_DIR / 'singlerun' / '2026-07-19' / 'cnt_carbon24'

if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print('downloading', f['name'], '...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()
print('checkpoints:', [c.name for c in MODEL_PATH.glob('*.ckpt')])

In [ ]:
# --- Load model & bind the SCIGEN sampler --------------------------------------
import torch
import hydra
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

def load_model_for_inference(model_path, device='cpu'):
    model_path = Path(model_path)
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')
    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'), ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))
    model = model.to(device)
    model.eval()
    return model, cfg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model_for_inference(MODEL_PATH, device=device)

from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)
print('model ready on', device)

## Generation settings

- `SC_TYPE='cnt'` pins a full CNT wall. Default chiralities are sampled from
  `CARBONTUBE_DEFAULTS['chirality_options']` = (3,3), (4,4), (5,5), (4,0), (5,0)
  → 12–20 wall atoms per axial period (fits the carbon_24 atom-count ceiling of 24).
  Fix a single chirality by editing that list in `script/sc_utils.py`, or wire a CNT
  database into `carbontube_params_from_db` in `script/gen_utils.py`.
- `DATASET='carbon_24'` makes every generated atom carbon and draws atom counts
  from the carbon_24 distribution. `SC_TYPE='ntb'` + `ELEMENT` in the magnetic
  metals reproduces the generalized-nanotube mode.
- `reduced_mask=False` is required: the wall is pinned per-coordinate, `mask_x (N,3)`.

In [ ]:
SC_TYPE = 'cnt'        # 'cnt' carbon nanotube | 'ntb' generalized tube | 'van'
ELEMENT = 'C'          # known species ('cnt' forces C regardless)
DATASET = 'carbon_24'  # all-carbon atom types + carbon atom-count distribution
BATCH_SIZE = 4
NUM_BATCHES = 1
FRAC_Z = 0.5           # axial phase of the wall (any value in [0,1) is fine)
STEP_LR = 5e-6
SEED = 42

SC_NATM_RANGE = {'cnt': [1, 24], 'ntb': [4, 24], 'van': [1, 20]}
natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])
total_structures = BATCH_SIZE * NUM_BATCHES
print(f'{SC_TYPE}: {total_structures} structures, natm_range {natm_range}')

In [ ]:
# --- Build the constrained dataset & run reverse diffusion ---------------------
from tqdm.auto import tqdm
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset

test_set = SampleDataset(
    dataset=DATASET,
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=[ELEMENT],
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

all_frac_coords, all_atom_types, all_lattices = [], [], []
all_num_atoms, all_num_known = [], []
for batch in tqdm(test_loader, desc='Generating structures'):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)
    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        raw_types = raw_types.argmax(dim=-1) + 1
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)
print('generated:', num_atoms.tolist(), 'atoms per structure (known wall:', num_known.tolist(), ')')

In [ ]:
# --- Convert to pymatgen structures --------------------------------------------
import numpy as np
from pymatgen.core.lattice import Lattice
from pymatgen.core.structure import Structure
from sc_utils import chemical_symbols

def lattices_to_params(lat):
    """(3,3) lattice matrix -> (lengths[3], angles_deg[3])."""
    lengths = np.linalg.norm(lat, axis=1)
    angles = np.zeros(3)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos = np.dot(lat[j], lat[k]) / (lengths[j] * lengths[k])
        angles[i] = np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
    return lengths, angles

structures = []
start = 0
for i in range(num_atoms.shape[0]):
    n_i = int(num_atoms[i])
    coords_i = frac_coords[start:start + n_i].numpy()
    types_i = atom_types[start:start + n_i].numpy()
    start += n_i
    lengths_i, angles_i = lattices_to_params(lattices[i].numpy())
    species = [chemical_symbols[int(t)] for t in types_i]
    try:
        structure = Structure(
            Lattice.from_parameters(*lengths_i, *angles_i),
            species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
    except Exception as e:
        structures.append(None)
        print(f'structure {i}: conversion failed ({e})')
print(sum(s is not None for s in structures), '/', len(structures), 'structures converted')

In [ ]:
# --- Quick 3D look at one candidate --------------------------------------------
import matplotlib.pyplot as plt

s0 = next((s for s in structures if s is not None), None)
if s0 is not None:
    xyz = s0.cart_coords
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], s=60)
    ax.set_title(s0.composition.reduced_formula)
    plt.show()

In [ ]:
# --- Export CIF files ----------------------------------------------------------
output_dir = Path.cwd().parent / 'NTGEN_generation' / 'generated_cifs'
output_dir.mkdir(parents=True, exist_ok=True)
for i, structure in enumerate(structures):
    if structure is None:
        continue
    formula = structure.composition.reduced_formula
    cif_path = output_dir / f'{SC_TYPE}_{formula}_{i:03d}.cif'
    structure.to(filename=str(cif_path), fmt='cif')
print('wrote CIFs to', output_dir)

## Retraining a carbon CSPDiffusion checkpoint (carbon_24)

The notebook above uses the public **mp_20** diffusion weights. For carbon-native
generation, retrain on **carbon_24** (data + config are already scaffolded in
NTGEN-edit — see `RETRAIN_CARBON.md` there for the full recipe):

**Prerequisites** (GPU machine): a Python env with `torch`, `torch_geometric`,
`torch_scatter`, `pytorch_lightning`, `hydra-core`, `pymatgen`, `p_tqdm`; the
`scigen -> ntgent` symlink; a `.env` (or exported vars) with `PROJECT_ROOT`
(NTGEN-edit path), `HYDRA_JOBS` (output dir), `WANDB_DIR`.

```bash
cd models/NTGEN-edit
python scigen/run.py data=carbon_24 model=diffusion_w_type expname=cnt_carbon24
```

- First run converts `data/carbon_24/{train,val,test}.csv` (CIF strings) into
  cached `*_ori.pt` tensors, then trains ~1000 epochs (hours–days on one GPU).
- Outputs land in `${HYDRA_JOBS}/singlerun/<date>/cnt_carbon24/`:
  `epoch=*.ckpt`, `hparams.yaml`, `lattice_scaler.pt`, `prop_scaler.pt` —
  exactly the bundle `load_model_for_inference` expects.
- Then set `MODEL_PATH` (checkpoint cell above) to that directory and rerun.